<a href="https://colab.research.google.com/github/priyaaaww/ITA06_LAB_EXP/blob/main/EXP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import pandas as pd
from google.colab import files

# Step 1: Load CSV file without repeated upload
filename = "/content/Covid Data.csv"

if not os.path.exists('/content/Covid Data.csv'):
    uploaded = files.upload()
    filename = next(iter(uploaded))

# Step 2: Read CSV file
data = pd.read_csv(filename, on_bad_lines="skip")

print("First 5 rows of the dataset:")
print(data.head())

# Step 3: Select attributes and target
attributes = ["USMER", "SEX", "PATIENT_TYPE", "PNEUMONIA"]
target = "CLASIFFICATION_FINAL"

if target not in data.columns:
    raise ValueError("Target column not found. Check the CSV column names.")

attributes = [a for a in attributes if a in data.columns]
data = data[attributes + [target]].dropna().copy()

# Step 4: Convert target to binary classes
data["Class"] = data[target].apply(
    lambda x: "Yes" if str(x).strip() == "3" else "No"
)

# Keep a small sample for demonstration
data = data.head(20)

X = data[attributes].astype(str).values
y = data["Class"].values
n = len(attributes)

# Step 5: Initialize boundaries
S = [["Ø"] * n]
G = [["?" for _ in range(n)]]

# Step 6: Candidate-Elimination algorithm
for x, label in zip(X, y):

    if label == "Yes":
        # Remove G hypotheses that do not cover positive example
        G = [
            g for g in G
            if all(g[i] == "?" or g[i] == x[i]
                   for i in range(n))
        ]

        # Generalize S minimally
        updated_S = []
        for s in S:
            if all(s[i] in ["Ø", "?", x[i]] for i in range(n)):
                h = s.copy()
                for i in range(n):
                    if h[i] == "Ø":
                        h[i] = x[i]
                    elif h[i] != x[i]:
                        h[i] = "?"
                updated_S.append(h)

        S = updated_S

    else:
        # Remove S hypotheses that cover negative example
        S = [
            s for s in S
            if not all(s[i] == "?" or s[i] == x[i]
                       for i in range(n))
        ]

        # Specialize G minimally
        new_G = []
        domains = [
            sorted(data[a].astype(str).unique())
            for a in attributes
        ]

        for g in G:
            if all(g[i] == "?" or g[i] == x[i]
                   for i in range(n)):
                for i in range(n):
                    if g[i] == "?":
                        for value in domains[i]:
                            if value != x[i]:
                                h = g.copy()
                                h[i] = value

                                if any(
                                    all(
                                        s[j] == "Ø"
                                        or s[j] == "?"
                                        or h[j] == "?"
                                        or s[j] == h[j]
                                        for j in range(n)
                                    )
                                    for s in S
                                ):
                                    new_G.append(h)
            else:
                new_G.append(g)

        G = new_G

    # Remove duplicates
    S = [list(h) for h in dict.fromkeys(map(tuple, S))]
    G = [list(h) for h in dict.fromkeys(map(tuple, G))]

# Step 7: Display results
print("\nAttributes:", attributes)
print("\nSpecific Boundary (S):")
for h in S:
    print(h)

print("\nGeneral Boundary (G):")
for h in G:
    print(h)

if not S or not G:
    print("\nNo consistent version space found for this sample.")

First 5 rows of the dataset:
   USMER  MEDICAL_UNIT  SEX  PATIENT_TYPE   DATE_DIED  INTUBED  PNEUMONIA  \
0      2             1    1             1  03/05/2020       97          1   
1      2             1    2             1  03/06/2020       97          1   
2      2             1    2             2  09/06/2020        1          2   
3      2             1    1             1  12/06/2020       97          2   
4      2             1    2             1  21/06/2020       97          2   

   AGE  PREGNANT  DIABETES  ...  ASTHMA  INMSUPR  HIPERTENSION  OTHER_DISEASE  \
0   65         2         2  ...       2        2             1              2   
1   72        97         2  ...       2        2             1              2   
2   55        97         1  ...       2        2             2              2   
3   53         2         2  ...       2        2             2              2   
4   68        97         1  ...       2        2             1              2   

   CARDIOVASCULAR  OB